In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📊 GraphQL API Design & Implementation Guide

**Building flexible, efficient APIs with GraphQL for modern applications**

This guide covers:
- GraphQL fundamentals vs REST
- Schema design and types
- Query optimization
- Mutations and subscriptions
- Authentication and authorization
- Caching strategies
- Performance monitoring
- Security best practices
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. GraphQL Basics

### GraphQL vs REST
```
REST API:
GET /users/123
└─ Returns all user fields (over-fetching)

GET /users/123/posts
└─ Requires separate request (under-fetching)

GraphQL:
{
  user(id: 123) {
    name
    email
    posts {
      title
      createdAt
    }
  }
}
└─ Single request, exact fields needed
```

### Schema Definition
```graphql
# Define types
type User {
  id: ID!
  name: String!
  email: String!
  posts: [Post!]!
  createdAt: DateTime!
}

type Post {
  id: ID!
  title: String!
  content: String!
  author: User!
  likes: Int!
  createdAt: DateTime!
}

# Root queries
type Query {
  user(id: ID!): User
  users(limit: Int = 10): [User!]!
  post(id: ID!): Post
}

# Mutations
type Mutation {
  createUser(name: String!, email: String!): User!
  updateUser(id: ID!, name: String): User
  deleteUser(id: ID!): Boolean!
  createPost(title: String!, content: String!): Post!
}

# Subscriptions (real-time)
type Subscription {
  userCreated: User!
  postLiked(postId: ID!): Int!
}
```

### Resolver Implementation (Python/Strawberry)
```python
import strawberry
from datetime import datetime
from typing import List

@strawberry.type
class User:
    id: strawberry.ID
    name: str
    email: str
    created_at: datetime

@strawberry.type
class Post:
    id: strawberry.ID
    title: str
    content: str
    author_id: strawberry.ID
    likes: int
    created_at: datetime

@strawberry.type
class Query:
    @strawberry.field
    async def user(self, id: strawberry.ID) -> User:
        """Fetch single user"""
        user = await db.query_one("SELECT * FROM users WHERE id = ?", id)
        return User(**user)
    
    @strawberry.field
    async def users(self, limit: int = 10) -> List[User]:
        """Fetch multiple users"""
        users = await db.query("SELECT * FROM users LIMIT ?", limit)
        return [User(**u) for u in users]

@strawberry.type
class Mutation:
    @strawberry.mutation
    async def create_user(self, name: str, email: str) -> User:
        """Create new user"""
        result = await db.execute(
            "INSERT INTO users (name, email) VALUES (?, ?)",
            (name, email)
        )
        return User(
            id=result.lastrowid,
            name=name,
            email=email,
            created_at=datetime.now()
        )

schema = strawberry.Schema(query=Query, mutation=Mutation)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Schema Design Patterns

### Pagination
```graphql
type Query {
  users(first: Int, after: String): UserConnection!
}

type UserConnection {
  edges: [UserEdge!]!
  pageInfo: PageInfo!
}

type UserEdge {
  node: User!
  cursor: String!
}

type PageInfo {
  hasNextPage: Boolean!
  endCursor: String
}
```

```python
# Cursor-based pagination (efficient for large datasets)
def get_users_with_pagination(first: int, after: str = None):
    query = "SELECT * FROM users"
    
    if after:
        # Decode cursor to get offset
        offset = decode_cursor(after)
        query += f" WHERE id > {offset}"
    
    query += f" LIMIT {first + 1}"
    
    users = db.query(query)
    
    has_next = len(users) > first
    users = users[:first]
    
    edges = [
        {
            'node': user,
            'cursor': encode_cursor(user['id'])
        }
        for user in users
    ]
    
    return {
        'edges': edges,
        'pageInfo': {
            'hasNextPage': has_next,
            'endCursor': edges[-1]['cursor'] if edges else None
        }
    }
```

### Batching (N+1 Prevention)
```python
from dataloader import DataLoader

# DataLoader batches queries
async def batch_load_authors(post_ids):
    """Load multiple authors in one query"""
    authors = await db.query(
        "SELECT * FROM users WHERE id IN (?)",
        (post_ids,)
    )
    # Return in same order as requested IDs
    return [authors_by_id.get(pid) for pid in post_ids]

author_loader = DataLoader(batch_load_authors)

# Resolver uses loader
@strawberry.field
async def author(self) -> User:
    return await author_loader.load(self.author_id)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Security & Performance

### Authentication
```python
# Verify JWT in context
@strawberry.type
class Query:
    @strawberry.field
    async def me(self, info) -> User:
        """Get current authenticated user"""
        token = info.context.get('authorization')
        
        if not token:
            raise Exception("Not authenticated")
        
        user_id = verify_jwt(token)
        return await db.get_user(user_id)
```

### Authorization
```graphql
# Schema directives for authorization
directive @auth(requires: String!) on FIELD_DEFINITION

type Query {
  users: [User!]! @auth(requires: "ADMIN")
  me: User! @auth(requires: "USER")
}
```

### Query Complexity Analysis
```python
# Prevent expensive queries
class ComplexityChecker:
    def __init__(self, max_complexity=1000):
        self.max_complexity = max_complexity
    
    def get_field_complexity(self, field_name, args):
        """Assign complexity to each field"""
        complexities = {
            'users': 5 * args.get('limit', 10),  # Linear with limit
            'posts': 2 * args.get('limit', 10),
            'comments': 1,
        }
        return complexities.get(field_name, 1)
    
    def calculate_query_complexity(self, document):
        """Calculate total query complexity"""
        total = 0
        # Walk AST, sum field complexities
        return total
    
    def validate(self, query_document):
        complexity = self.calculate_query_complexity(query_document)
        
        if complexity > self.max_complexity:
            raise Exception(
                f"Query too complex: {complexity} > {self.max_complexity}"
            )
```

### Caching
```python
# Cache at resolver level
@strawberry.field
async def user(self, id: strawberry.ID) -> User:
    cache_key = f"user:{id}"
    
    # Check cache
    cached = await redis.get(cache_key)
    if cached:
        return User(**json.loads(cached))
    
    # Fetch from DB
    user = await db.get_user(id)
    
    # Cache for 1 hour
    await redis.setex(cache_key, 3600, json.dumps(user))
    
    return user
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 4. Subscriptions (Real-Time)

```python
import asyncio

@strawberry.type
class Subscription:
    @strawberry.subscription
    async def user_created(self) -> User:
        """Subscribe to new user creations"""
        async for user in user_events.subscribe('user.created'):
            yield user

# Publishing events
async def create_user(name: str, email: str):
    user = await db.create_user(name, email)
    await user_events.publish('user.created', user)
    return user
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. GraphQL Best Practices

- **Query complexity limits**: Prevent expensive queries
- **Depth limiting**: Max query depth (e.g., 10)
- **Rate limiting**: Queries per user/hour
- **Batch loading**: Prevent N+1 queries
- **Caching**: At resolver and HTTP level
- **Error handling**: Standardized error format
- **Monitoring**: Query performance metrics
- **Documentation**: Auto-generated from schema
</MySCode.Cell>
```